# Adaptive Closed-Loop Binaural Beat Therapy for Migraine

This notebook implements a **real-time adaptive system** that:
1. **Initializes** binaural beat frequency based on patient EEG and demographics
2. **Monitors** EEG changes every minute (delta, theta, alpha, beta, gamma)
3. **Updates weights** dynamically based on brain response
4. **Adjusts frequency** to drive brain toward target therapeutic state
5. **Incorporates clinical feedback** (pain level, subjective improvement)

---

## Mathematical Framework

### Adaptive Frequency Update
$$f_b(t+\Delta t) = f_b(t) + \alpha \cdot \left(\sum_{k} w_k(t) \cdot \Delta P_k(t)\right) + \beta \cdot E(t)$$

### Weight Update (Online Learning)
$$w_k(t+\Delta t) = w_k(t) + \eta \cdot (P_{k,target}(t) - P_{k,observed}(t))$$

### Simple Control Law (Alternative)
$$f_b(t+\Delta t) = f_b(t) + K \cdot (\text{target alpha} - \text{observed alpha})$$

---

In [ ]:
# Import libraries
import sys
sys.path.insert(0, 'src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.io import wavfile
from scipy import signal

# Import custom modules
from data_loader import load_clinical_data, load_eeg_file
from feature_extraction import extract_psd_features
from classifier import load_model, predict

# Configure plotting
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)
np.random.seed(42)

print("✓ Libraries imported successfully")

## 1. Define Adaptive System Parameters

In [ ]:
# System hyperparameters
LEARNING_RATES = {
    'alpha': 0.05,      # α: EEG feedback learning rate
    'beta': 0.10,       # β: Clinical feedback learning rate  
    'eta': 0.03         # η: Weight update learning rate
}

# Control parameters
UPDATE_INTERVAL = 60        # seconds (1 minute)
TREATMENT_DURATION = 20     # minutes
FREQUENCY_BOUNDS = (4, 13)  # Hz (theta to alpha range)

# Target EEG state for migraine therapy (normalized power ratios)
TARGET_EEG_STATE = {
    'delta': 0.15,   # Low delta (avoid drowsiness)
    'theta': 0.20,   # Moderate theta (relaxation)
    'alpha': 0.40,   # HIGH alpha (therapeutic target)
    'beta': 0.20,    # Moderate beta (calm alertness)
    'gamma': 0.05    # Low gamma (reduce hyperexcitability)
}

print("Adaptive System Configuration:")
print(f"  Update interval: {UPDATE_INTERVAL} seconds")
print(f"  Treatment duration: {TREATMENT_DURATION} minutes")
print(f"  Frequency range: {FREQUENCY_BOUNDS[0]}-{FREQUENCY_BOUNDS[1]} Hz")
print(f"\nTarget EEG State (Therapeutic):")
for band, target in TARGET_EEG_STATE.items():
    print(f"  {band.capitalize()}: {target:.0%}")

## 2. Initialize Patient Profile

Load patient EEG data and extract baseline frequency band powers.

In [ ]:
def extract_band_powers(raw):
    """
    Extract normalized power in each frequency band
    
    Returns:
        dict: Band powers normalized to sum to 1.0
    """
    data = raw.get_data()
    sfreq = raw.info['sfreq']
    
    # Get EEG channels only
    ch_types = raw.get_channel_types()
    eeg_indices = [i for i, ch_type in enumerate(ch_types) if ch_type == 'eeg']
    if len(eeg_indices) == 0:
        eeg_indices = list(range(min(128, data.shape[0])))
    
    eeg_data = data[eeg_indices, :]
    
    # Define frequency bands
    bands = {
        'delta': (0.5, 4),
        'theta': (4, 8),
        'alpha': (8, 13),
        'beta': (13, 30),
        'gamma': (30, 50)
    }
    
    # Calculate average power across all channels
    band_powers = {}
    
    for band_name, (low, high) in bands.items():
        band_power_sum = 0
        for ch_idx in range(eeg_data.shape[0]):
            freqs, psd = signal.welch(eeg_data[ch_idx, :], sfreq, 
                                     nperseg=min(2048, eeg_data.shape[1]))
            idx_band = np.logical_and(freqs >= low, freqs <= high)
            band_power_sum += np.mean(psd[idx_band])
        
        band_powers[band_name] = band_power_sum / eeg_data.shape[0]
    
    # Normalize to proportions (sum = 1.0)
    total_power = sum(band_powers.values())
    normalized_powers = {k: v/total_power for k, v in band_powers.items()}
    
    return normalized_powers


# Load patient data
PATIENT_ID = 'M1_1'  # Migraine with Aura patient

print(f"Loading patient: {PATIENT_ID}\n")

# Load clinical data
clinical_df = load_clinical_data()
base_id = PATIENT_ID.split('_')[0]
patient_info = clinical_df[clinical_df['P#'] == base_id].iloc[0]

# Load EEG and extract baseline
raw = load_eeg_file(PATIENT_ID, 'resting', verbose=False)
baseline_eeg = extract_band_powers(raw)

# Display patient profile
print("Patient Profile:")
print(f"  ID: {PATIENT_ID}")
print(f"  Age: {patient_info['Age']}")
print(f"  Gender: {patient_info['Gender']}")
print(f"  Migraine Type: {'Aura' if patient_info['Aura?'] == 'Yes' else 'Non-Aura'}")
print(f"\nBaseline EEG State:")
for band, power in baseline_eeg.items():
    print(f"  {band.capitalize()}: {power:.2%}")

## 3. Initialize Weights and Starting Frequency

Weights represent the influence of each factor on frequency adjustment.

In [ ]:
def initialize_weights(patient_info, baseline_eeg):
    """
    Initialize weights based on patient characteristics
    
    Returns:
        dict: Initial weights for each factor
    """
    weights = {}
    
    # EEG band weights (based on deviation from target)
    for band in ['delta', 'theta', 'alpha', 'beta', 'gamma']:
        deviation = TARGET_EEG_STATE[band] - baseline_eeg[band]
        # Positive deviation means we need to increase that band
        weights[f'w_{band}'] = 0.5 + deviation  # Start around 0.5, adjust by deviation
    
    # Clinical factor weights
    age = patient_info['Age']
    weights['w_age'] = 1.0 - (age - 20) / 60  # Younger = higher weight (0.5-1.0)
    weights['w_age'] = np.clip(weights['w_age'], 0.3, 1.0)
    
    weights['w_gender'] = 0.9 if patient_info['Gender'] == 'Female' else 0.7
    
    weights['w_migraine_type'] = 1.2 if patient_info['Aura?'] == 'Yes' else 0.8
    
    return weights


def calculate_initial_frequency(baseline_eeg, weights, patient_info):
    """
    Calculate starting binaural beat frequency
    
    Returns:
        float: Initial frequency in Hz
    """
    # Base frequency depends on migraine type
    if patient_info['Aura?'] == 'Yes':
        base_freq = 10.0  # Alpha band (calming hyperexcitability)
    else:
        base_freq = 7.5   # Theta-alpha transition
    
    # Adjust based on alpha deficit
    alpha_deficit = TARGET_EEG_STATE['alpha'] - baseline_eeg['alpha']
    
    # If alpha is low, push frequency higher to stimulate it
    frequency_adjustment = alpha_deficit * 5.0  # Scale factor
    
    initial_freq = base_freq + frequency_adjustment
    
    # Constrain to valid range
    initial_freq = np.clip(initial_freq, FREQUENCY_BOUNDS[0], FREQUENCY_BOUNDS[1])
    
    return initial_freq


# Initialize system
weights = initialize_weights(patient_info, baseline_eeg)
initial_frequency = calculate_initial_frequency(baseline_eeg, weights, patient_info)

print("\nInitial Weights:")
for key, value in weights.items():
    print(f"  {key}: {value:.3f}")

print(f"\nInitial Binaural Beat Frequency: {initial_frequency:.2f} Hz")
print(f"  → Targeting {'Alpha' if initial_frequency > 8 else 'Theta-Alpha'} band")

## 4. Simulate Real-Time EEG Monitoring

In a real system, you would measure EEG every minute. Here we **simulate** the brain's response to therapy.

In [ ]:
def simulate_eeg_response(current_eeg, target_eeg, current_frequency, noise_level=0.05):
    """
    Simulate how EEG changes in response to binaural beat stimulation
    
    In reality, this would be measured from the patient's EEG headset.
    Here we simulate a gradual shift toward the target state with some noise.
    
    Args:
        current_eeg: Current EEG band powers
        target_eeg: Target EEG state
        current_frequency: Current binaural beat frequency
        noise_level: Random variation in measurement
        
    Returns:
        dict: Simulated new EEG state
    """
    new_eeg = {}
    
    # Determine which band is being stimulated
    if 4 <= current_frequency < 8:
        stimulated_band = 'theta'
    elif 8 <= current_frequency < 13:
        stimulated_band = 'alpha'
    else:
        stimulated_band = 'beta'
    
    for band in current_eeg.keys():
        # Move toward target (convergence rate)
        convergence_rate = 0.15 if band == stimulated_band else 0.05
        
        # Calculate change
        target_power = target_eeg[band]
        current_power = current_eeg[band]
        
        change = convergence_rate * (target_power - current_power)
        
        # Add biological noise
        noise = np.random.normal(0, noise_level * current_power)
        
        new_eeg[band] = current_power + change + noise
    
    # Renormalize (powers must sum to 1.0)
    total = sum(new_eeg.values())
    new_eeg = {k: v/total for k, v in new_eeg.items()}
    
    return new_eeg


def simulate_clinical_feedback(time_step, eeg_improvement):
    """
    Simulate patient's subjective pain/discomfort rating
    
    Args:
        time_step: Current time step (0 = start)
        eeg_improvement: How much EEG has improved toward target
        
    Returns:
        float: Clinical score (-1 to +1, where +1 = significant improvement)
    """
    # Early treatment may cause slight discomfort
    if time_step < 3:
        base_score = -0.2
    # Mid treatment shows improvement
    elif time_step < 10:
        base_score = 0.3 + (time_step / 10) * 0.4
    # Later treatment plateaus
    else:
        base_score = 0.7
    
    # Adjust by how well EEG is improving
    score = base_score + eeg_improvement * 0.3
    
    # Add some noise
    score += np.random.normal(0, 0.1)
    
    return np.clip(score, -1, 1)


print("✓ EEG simulation functions defined")

## 5. Implement Adaptive Update Equations

In [ ]:
def calculate_eeg_changes(current_eeg, previous_eeg):
    """
    Calculate ΔP_k(t) = P_k(t+Δt) - P_k(t) for each band
    """
    delta_P = {}
    for band in current_eeg.keys():
        delta_P[band] = current_eeg[band] - previous_eeg[band]
    return delta_P


def update_weights(weights, current_eeg, target_eeg, learning_rate):
    """
    Update weights using online learning:
    w_k(t+Δt) = w_k(t) + η·(P_target - P_observed)
    
    Args:
        weights: Current weight dictionary
        current_eeg: Observed EEG state
        target_eeg: Target EEG state
        learning_rate: η (eta)
        
    Returns:
        dict: Updated weights
    """
    new_weights = weights.copy()
    
    # Update EEG band weights
    for band in ['delta', 'theta', 'alpha', 'beta', 'gamma']:
        error = target_eeg[band] - current_eeg[band]
        new_weights[f'w_{band}'] += learning_rate * error
        
        # Constrain weights to reasonable range
        new_weights[f'w_{band}'] = np.clip(new_weights[f'w_{band}'], 0.1, 2.0)
    
    return new_weights


def update_frequency(current_freq, weights, delta_P, clinical_score, learning_rates):
    """
    Main adaptive frequency update equation:
    f_b(t+Δt) = f_b(t) + α·(Σw_k·ΔP_k) + β·E(t)
    
    Args:
        current_freq: Current binaural beat frequency
        weights: Current weights
        delta_P: Changes in EEG band powers
        clinical_score: Patient feedback score
        learning_rates: Dict with 'alpha' and 'beta'
        
    Returns:
        float: Updated frequency
    """
    alpha = learning_rates['alpha']
    beta = learning_rates['beta']
    
    # Calculate weighted sum of EEG changes
    weighted_eeg_change = 0
    for band in ['delta', 'theta', 'alpha', 'beta', 'gamma']:
        weighted_eeg_change += weights[f'w_{band}'] * delta_P[band]
    
    # Apply update equation
    frequency_change = alpha * weighted_eeg_change + beta * clinical_score
    
    new_freq = current_freq + frequency_change
    
    # Constrain to valid frequency range
    new_freq = np.clip(new_freq, FREQUENCY_BOUNDS[0], FREQUENCY_BOUNDS[1])
    
    return new_freq


print("✓ Adaptive update equations implemented")

## 6. Run Closed-Loop Treatment Simulation

Simulate 20 minutes of adaptive treatment with 1-minute updates.

In [ ]:
# Initialize tracking
num_steps = TREATMENT_DURATION  # 20 steps (1 per minute)

history = {
    'time': [],
    'frequency': [],
    'clinical_score': [],
    'eeg': {'delta': [], 'theta': [], 'alpha': [], 'beta': [], 'gamma': []},
    'weights': {k: [] for k in weights.keys()}
}

# Starting state
current_frequency = initial_frequency
current_eeg = baseline_eeg.copy()
current_weights = weights.copy()

print("=" * 70)
print("CLOSED-LOOP ADAPTIVE TREATMENT SIMULATION")
print("=" * 70)
print(f"Patient: {PATIENT_ID}")
print(f"Duration: {TREATMENT_DURATION} minutes")
print(f"Update interval: {UPDATE_INTERVAL} seconds\n")

# Run simulation loop
for step in range(num_steps):
    time_minutes = step
    
    # Record current state
    history['time'].append(time_minutes)
    history['frequency'].append(current_frequency)
    for band, power in current_eeg.items():
        history['eeg'][band].append(power)
    for weight_name, weight_value in current_weights.items():
        history['weights'][weight_name].append(weight_value)
    
    # Display progress every 5 minutes
    if step % 5 == 0:
        print(f"\n[Minute {step}]")
        print(f"  Frequency: {current_frequency:.2f} Hz")
        print(f"  Alpha power: {current_eeg['alpha']:.2%} (target: {TARGET_EEG_STATE['alpha']:.2%})")
    
    # === ADAPTIVE FEEDBACK LOOP ===
    
    # 1. Deliver binaural beat at current_frequency (simulated)
    #    In real system: Generate and play audio for 60 seconds
    
    # 2. Measure new EEG state (simulated)
    previous_eeg = current_eeg.copy()
    current_eeg = simulate_eeg_response(current_eeg, TARGET_EEG_STATE, current_frequency)
    
    # 3. Calculate EEG changes
    delta_P = calculate_eeg_changes(current_eeg, previous_eeg)
    
    # 4. Get clinical feedback (simulated)
    eeg_improvement = (TARGET_EEG_STATE['alpha'] - current_eeg['alpha']) / TARGET_EEG_STATE['alpha']
    clinical_score = simulate_clinical_feedback(step, eeg_improvement)
    history['clinical_score'].append(clinical_score)
    
    # 5. Update weights (online learning)
    current_weights = update_weights(current_weights, current_eeg, 
                                    TARGET_EEG_STATE, LEARNING_RATES['eta'])
    
    # 6. Update frequency (adaptive control)
    current_frequency = update_frequency(current_frequency, current_weights, 
                                        delta_P, clinical_score, LEARNING_RATES)

print("\n" + "=" * 70)
print("TREATMENT COMPLETE")
print("=" * 70)
print(f"\nFinal State:")
print(f"  Frequency: {current_frequency:.2f} Hz")
print(f"  Alpha power: {current_eeg['alpha']:.2%} (target: {TARGET_EEG_STATE['alpha']:.2%})")
print(f"  Improvement: {(current_eeg['alpha'] - baseline_eeg['alpha']) / baseline_eeg['alpha']:.1%}")

## 7. Visualize Adaptive Changes

In [ ]:
# Create comprehensive visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
time = history['time']

# 1. Frequency adaptation over time
ax1 = axes[0, 0]
ax1.plot(time, history['frequency'], linewidth=2, color='#e74c3c', marker='o', markersize=4)
ax1.axhline(y=initial_frequency, color='gray', linestyle='--', alpha=0.5, label='Initial')
ax1.fill_between([FREQUENCY_BOUNDS[0], FREQUENCY_BOUNDS[1]], 0, 15, alpha=0.1, color='green')
ax1.set_xlabel('Time (minutes)', fontsize=12)
ax1.set_ylabel('Binaural Beat Frequency (Hz)', fontsize=12)
ax1.set_title('Adaptive Frequency Adjustment', fontsize=14, fontweight='bold')
ax1.grid(alpha=0.3)
ax1.legend()

# 2. EEG band power evolution
ax2 = axes[0, 1]
colors = {'delta': '#3498db', 'theta': '#9b59b6', 'alpha': '#2ecc71', 
          'beta': '#f39c12', 'gamma': '#e74c3c'}
for band, band_history in history['eeg'].items():
    ax2.plot(time, np.array(band_history) * 100, linewidth=2, 
             label=band.capitalize(), color=colors[band], marker='o', markersize=3)
    # Plot target
    ax2.axhline(y=TARGET_EEG_STATE[band] * 100, color=colors[band], 
                linestyle='--', alpha=0.3)

ax2.set_xlabel('Time (minutes)', fontsize=12)
ax2.set_ylabel('Band Power (%)', fontsize=12)
ax2.set_title('EEG Band Power Evolution', fontsize=14, fontweight='bold')
ax2.legend(loc='best')
ax2.grid(alpha=0.3)

# 3. Alpha power focusing (most important)
ax3 = axes[1, 0]
alpha_history = np.array(history['eeg']['alpha']) * 100
ax3.plot(time, alpha_history, linewidth=3, color='#2ecc71', marker='o', markersize=5, label='Observed')
ax3.axhline(y=TARGET_EEG_STATE['alpha'] * 100, color='red', linestyle='--', 
            linewidth=2, label='Target')
ax3.fill_between(time, alpha_history, TARGET_EEG_STATE['alpha'] * 100, 
                 where=(alpha_history < TARGET_EEG_STATE['alpha'] * 100),
                 alpha=0.3, color='red', label='Deficit')
ax3.set_xlabel('Time (minutes)', fontsize=12)
ax3.set_ylabel('Alpha Power (%)', fontsize=12)
ax3.set_title('Alpha Band Convergence (Therapeutic Target)', fontsize=14, fontweight='bold')
ax3.legend()
ax3.grid(alpha=0.3)

# 4. Clinical feedback score
ax4 = axes[1, 1]
clinical_history = history['clinical_score']
colors_clinical = ['red' if s < 0 else 'orange' if s < 0.3 else 'green' for s in clinical_history]
ax4.bar(time, clinical_history, color=colors_clinical, alpha=0.7, edgecolor='black')
ax4.axhline(y=0, color='black', linewidth=1)
ax4.set_xlabel('Time (minutes)', fontsize=12)
ax4.set_ylabel('Clinical Score', fontsize=12)
ax4.set_title('Patient Feedback (Simulated)', fontsize=14, fontweight='bold')
ax4.set_ylim([-1, 1])
ax4.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('output/adaptive_treatment_visualization.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Visualization saved to: output/adaptive_treatment_visualization.png")

## 8. Weight Evolution Analysis

In [ ]:
# Plot weight evolution over time
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# EEG band weights
ax1 = axes[0]
for band in ['delta', 'theta', 'alpha', 'beta', 'gamma']:
    weight_history = history['weights'][f'w_{band}']
    ax1.plot(time, weight_history, linewidth=2, label=band.capitalize(), 
             color=colors[band], marker='o', markersize=3)

ax1.set_xlabel('Time (minutes)', fontsize=12)
ax1.set_ylabel('Weight Value', fontsize=12)
ax1.set_title('EEG Band Weight Evolution (Online Learning)', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(alpha=0.3)

# Clinical factor weights
ax2 = axes[1]
for factor in ['w_age', 'w_gender', 'w_migraine_type']:
    if factor in history['weights']:
        weight_history = history['weights'][factor]
        ax2.plot(time, weight_history, linewidth=2, label=factor.replace('w_', '').replace('_', ' ').title(), 
                 marker='s', markersize=4)

ax2.set_xlabel('Time (minutes)', fontsize=12)
ax2.set_ylabel('Weight Value', fontsize=12)
ax2.set_title('Clinical Factor Weights', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('output/weight_evolution.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Weight evolution saved to: output/weight_evolution.png")

## 9. Generate Adaptive Treatment Report

In [ ]:
def generate_adaptive_report(patient_id, history, baseline_eeg, final_eeg, patient_info):
    """
    Generate comprehensive adaptive treatment report
    """
    report = []
    report.append("="*70)
    report.append("ADAPTIVE CLOSED-LOOP BINAURAL BEAT THERAPY REPORT")
    report.append("="*70)
    report.append("")
    report.append(f"Patient ID: {patient_id}")
    report.append(f"Date: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')}")
    report.append(f"Treatment Duration: {len(history['time'])} minutes")
    report.append("")
    report.append("-" * 70)
    report.append("PATIENT PROFILE")
    report.append("-" * 70)
    report.append(f"  Age: {patient_info['Age']}")
    report.append(f"  Gender: {patient_info['Gender']}")
    report.append(f"  Migraine Type: {'Aura' if patient_info['Aura?'] == 'Yes' else 'Non-Aura'}")
    report.append("")
    
    report.append("-" * 70)
    report.append("BASELINE vs FINAL EEG STATE")
    report.append("-" * 70)
    for band in ['delta', 'theta', 'alpha', 'beta', 'gamma']:
        baseline = baseline_eeg[band] * 100
        final = final_eeg[band] * 100
        change = final - baseline
        arrow = "↑" if change > 0 else "↓"
        report.append(f"  {band.capitalize():6s}: {baseline:5.1f}% → {final:5.1f}% ({arrow} {abs(change):4.1f}%)")
    report.append("")
    
    report.append("-" * 70)
    report.append("FREQUENCY ADAPTATION")
    report.append("-" * 70)
    report.append(f"  Initial Frequency: {history['frequency'][0]:.2f} Hz")
    report.append(f"  Final Frequency: {history['frequency'][-1]:.2f} Hz")
    report.append(f"  Total Adjustment: {history['frequency'][-1] - history['frequency'][0]:.2f} Hz")
    report.append(f"  Number of Updates: {len(history['frequency'])}")
    report.append("")
    
    report.append("-" * 70)
    report.append("THERAPEUTIC OUTCOME")
    report.append("-" * 70)
    alpha_improvement = (final_eeg['alpha'] - baseline_eeg['alpha']) / baseline_eeg['alpha'] * 100
    target_achievement = (final_eeg['alpha'] / TARGET_EEG_STATE['alpha']) * 100
    report.append(f"  Alpha Power Improvement: {alpha_improvement:+.1f}%")
    report.append(f"  Target Achievement: {min(target_achievement, 100):.1f}%")
    report.append(f"  Final Clinical Score: {history['clinical_score'][-1]:.2f} / 1.00")
    report.append("")
    
    report.append("-" * 70)
    report.append("ADAPTIVE SYSTEM PERFORMANCE")
    report.append("-" * 70)
    report.append(f"  Learning Rate (α): {LEARNING_RATES['alpha']}")
    report.append(f"  Clinical Weight (β): {LEARNING_RATES['beta']}")
    report.append(f"  Weight Update Rate (η): {LEARNING_RATES['eta']}")
    report.append(f"  Convergence: {min((final_eeg['alpha'] / TARGET_EEG_STATE['alpha']) * 100, 100):.0f}%")
    report.append("")
    
    report.append("-" * 70)
    report.append("CLINICAL INTERPRETATION")
    report.append("-" * 70)
    if alpha_improvement > 20:
        report.append("  ✓ EXCELLENT: Significant alpha enhancement achieved")
    elif alpha_improvement > 10:
        report.append("  ✓ GOOD: Moderate alpha improvement observed")
    else:
        report.append("  ⚠ MODEST: Limited alpha response, consider extended treatment")
    report.append("")
    
    report.append("=" * 70)
    report.append("")
    
    return "\n".join(report)


# Generate and save report
final_eeg = {band: history['eeg'][band][-1] for band in ['delta', 'theta', 'alpha', 'beta', 'gamma']}
report_text = generate_adaptive_report(PATIENT_ID, history, baseline_eeg, final_eeg, patient_info)

print(report_text)

# Save to file
report_path = f"output/{PATIENT_ID}_adaptive_treatment_report.txt"
Path(report_path).parent.mkdir(exist_ok=True, parents=True)
with open(report_path, 'w') as f:
    f.write(report_text)

print(f"\n✓ Report saved to: {report_path}")

## 10. Export Adaptive Treatment History

In [ ]:
# Create comprehensive DataFrame
df_history = pd.DataFrame({
    'Time_min': history['time'],
    'Frequency_Hz': history['frequency'],
    'Clinical_Score': history['clinical_score'],
    'Delta_%': np.array(history['eeg']['delta']) * 100,
    'Theta_%': np.array(history['eeg']['theta']) * 100,
    'Alpha_%': np.array(history['eeg']['alpha']) * 100,
    'Beta_%': np.array(history['eeg']['beta']) * 100,
    'Gamma_%': np.array(history['eeg']['gamma']) * 100,
})

# Add weight columns
for weight_name, weight_history in history['weights'].items():
    df_history[weight_name] = weight_history

# Save to CSV
csv_path = f"output/{PATIENT_ID}_adaptive_history.csv"
df_history.to_csv(csv_path, index=False)

print(f"✓ Treatment history saved to: {csv_path}")
print(f"\nPreview:")
display(df_history.head(10))

## 11. Summary & Next Steps

### ✅ What Was Achieved:

1. **Mathematical Model Implemented**:
   - Frequency update: $f_b(t+\Delta t) = f_b(t) + \alpha \cdot (\sum w_k \cdot \Delta P_k) + \beta \cdot E(t)$
   - Weight learning: $w_k(t+\Delta t) = w_k(t) + \eta \cdot (P_{target} - P_{observed})$

2. **Closed-Loop System**:
   - Real-time EEG monitoring (every 60 seconds)
   - Dynamic frequency adaptation
   - Online weight learning
   - Clinical feedback integration

3. **Personalization Factors**:
   - EEG band powers (delta, theta, alpha, beta, gamma)
   - Patient demographics (age, gender)
   - Migraine type (aura vs non-aura)
   - Clinical evaluation scores

---

### 🔬 For Real Implementation:

**Hardware Requirements:**
- 16-channel EEG headset (e.g., OpenBCI CytonDaisy)
- Real-time signal processing pipeline
- Audio delivery system (Bluetooth headphones)

**Software Integration:**
```python
# Replace simulation with real EEG acquisition
from brainflow import BoardShim, BrainFlowInputParams

# Real-time feature extraction
current_eeg = extract_band_powers_realtime(eeg_stream)

# Generate and play adaptive audio
audio = generate_binaural_beat(current_frequency, duration=60)
play_audio(audio)
```

**Clinical Validation:**
- Test with real migraine patients
- Compare adaptive vs static binaural beats
- Measure objective outcomes (pain scales, frequency of attacks)

---

### 📚 Research Papers for Reference:

1. **Neurofeedback & Migraine**: Stokes & Lappin (2010) - "Neurofeedback and biofeedback with 37 migraineurs"
2. **Closed-Loop BCI**: Zrenner et al. (2016) - "Closed-loop neuroscience and non-invasive brain stimulation"
3. **Adaptive Algorithms**: Vidaurre & Blankertz (2010) - "Towards a cure for BCI illiteracy"
4. **EEG-Based Control**: Ang et al. (2015) - "A large clinical study on the ability of stroke patients to use an EEG-based motor imagery brain-computer interface"

---